# Homework 1: Language Identification
11-411/11-611 Natural Language Processing (Fall 2026)

**Due:** Refer to Canvas and Gradescope for timeline.

**Submission:** Upload this single file named `HW1.ipynb` to Gradescope. Keep `train.tsv`, `test.tsv`, and `confusion_matrix.png` in the same directory while you work, but submit only the notebook unless your instructor says otherwise.

In this assignment, you will build a language identification classifier that distinguishes between six languages:

- [Hausa](https://en.wikipedia.org/wiki/Hausa_language)
- [Indonesian](https://en.wikipedia.org/wiki/Indonesian_language)
- [Manobo](https://en.wikipedia.org/wiki/Manobo_languages)
- [Nahuatl](https://en.wikipedia.org/wiki/Nahuatl)
- [Swahili](https://en.wikipedia.org/wiki/Swahili_language)
- [Tagalog](https://en.wikipedia.org/wiki/Tagalog_language)

All six are represented here using the [Latin script](https://en.wikipedia.org/wiki/Latin_script), with relatively few [diacritics](https://en.wikipedia.org/wiki/Diacritic), so simple script-based rules are not enough to distinguish them reliably.

A natural approach is to use character features. Individual character frequencies differ across languages, and short character sequences can capture useful orthographic and phonotactic patterns. Character $n$-grams therefore provide a compact feature representation for language identification. If $n$ becomes too large, however, the feature space becomes sparse and the model can overfit.

You will implement multinomial logistic regression (softmax regression) from scratch and use it to compare several character $n$-gram orders, inspect errors, and interpret learned feature weights.


## Requirements

You will:

1. Implement the gradient and training loop for multinomial logistic regression.
2. Implement inference for multinomial logistic regression.
3. Compare character $n$-gram orders $n \in \{1,2,3,4\}$ and select a model using macro-averaged F1.
4. Calculate and display a confusion matrix for a fixed trigram model.
5. Inspect a specific family of classification errors and formulate testable hypotheses.
6. Inspect feature weights and identify the most predictive character trigrams for each language.


## How to Use Jupyter Notebooks for This Assignment

This notebook mixes explanations, equations, implementation tasks, and short analyses. By the end of the assignment, you should be comfortable with character $n$-gram features, multinomial logistic regression, evaluation metrics, confusion matrices, and feature-weight interpretation.

### Types of cells
**Markdown cells** contain explanations, questions, equations, and response prompts.

**Code cells** contain provided code or places where you will implement a function or produce an output used by the grader.

Do not delete or reorder provided cells. You may use additional scratch cells at the end of the notebook while debugging, but your submitted notebook should run from top to bottom.

### Runtime volatility
If you restart your runtime or disconnect, variables and outputs in memory are lost. Save your notebook frequently and use **Run all** before submitting.

### Running on Google Colab
You may run the notebook in Google Colab. Upload `train.tsv`, `test.tsv`, and `confusion_matrix.png` into the same working directory as the notebook before running the data-loading cells.


## Important: Library Restrictions

**You must implement multinomial logistic regression from scratch using only the provided imports.**

You are **not allowed** to use:
- PyTorch, TensorFlow, JAX, or other deep-learning frameworks
- Scikit-learn or other machine-learning libraries
- External optimization libraries

You may use only the imported libraries below: `csv`, `random`, `numpy`, `numpy.typing`, `numpy.testing`, and `collections.Counter`.

## Imports
Do not change.


In [1]:
import csv
import random

import numpy as np
import numpy.typing as npt
import numpy.testing as testing

from collections import Counter

# Revert to legacy printing style
np.set_printoptions(legacy='1.25')

## Helper Functions
### Understanding Data Structures

Throughout this assignment, you'll work with these key data structures:

1. **Raw observations**: `list[tuple[str, str]]` - List of (language_name, document_text) pairs
2. **Processed observations**: `list[tuple[np.ndarray, np.ndarray]]` - List of (one_hot_label, feature_vector) pairs
3. **Weight matrix W**: `np.ndarray` shape `[K, N]` where K=number of languages, N=number of features
   - **Important**: Column 0 contains bias terms, so remember to prepend 1 to feature vectors
4. **Feature map**: `dict[str, int]` - Maps n-gram strings to feature indices
5. **Language map**: `dict[str, np.ndarray]` - Maps language names to one-hot vectors

### Metrics for Binary Classification

In [2]:
def precision(tp: int, fp: int) -> float:
    """Computes precision from true positives and false positives."""
    denom = tp + fp
    return tp / denom if denom else 0.0


def recall(tp: int, fn: int) -> float:
    """Computes recall from true positives and false negatives."""
    denom = tp + fn
    return tp / denom if denom else 0.0


def f_measure(beta: float, tp: int, fp: int, fn: int) -> float:
    """Computes F-beta from TP, FP, and FN."""
    p = precision(tp, fp)
    r = recall(tp, fn)
    denom = beta**2 * p + r
    return (1 + beta**2) * p * r / denom if denom else 0.0


def f1(tp: int, fp: int, fn: int) -> float:
    """Computes F1 from TP, FP, and FN."""
    return f_measure(1, tp, fp, fn)


### Micro-Averaged Metrics

In [3]:
def micro_precision(tp: "dict[int, int]", fp: "dict[int, int]") -> float:
    """Computes micro-averaged precision."""
    tp_sum = sum(tp.values())
    fp_sum = sum(fp.values())
    denom = tp_sum + fp_sum
    return tp_sum / denom if denom else 0.0


def micro_recall(tp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes micro-averaged recall."""
    tp_sum = sum(tp.values())
    fn_sum = sum(fn.values())
    denom = tp_sum + fn_sum
    return tp_sum / denom if denom else 0.0


def micro_f1(tp: "dict[int, int]", fp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes micro-averaged F1."""
    mp = micro_precision(tp, fp)
    mr = micro_recall(tp, fn)
    return 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0


### Macro-Averaged Metrics

In [4]:
def _metric_classes(*counters) -> list[int]:
    """Return the sorted union of class IDs observed in metric counters."""
    return sorted(set().union(*(counter.keys() for counter in counters)))


def macro_precision(tp: "dict[int, int]", fp: "dict[int, int]") -> float:
    """Computes macro-averaged precision."""
    classes = _metric_classes(tp, fp)
    return np.mean([precision(tp[c], fp[c]) for c in classes]) if classes else 0.0


def macro_recall(tp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes macro-averaged recall."""
    classes = _metric_classes(tp, fn)
    return np.mean([recall(tp[c], fn[c]) for c in classes]) if classes else 0.0


def macro_f1(tp: "dict[int, int]", fp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes macro-F1 as the mean of the per-class F1 scores."""
    classes = _metric_classes(tp, fp, fn)
    return np.mean([f1(tp[c], fp[c], fn[c]) for c in classes]) if classes else 0.0


## Preprocess Data

### Loading data from files

We first need to load data from the provided TSV files. Each file is two columns, the language of the document and the document text, separated by a tab (`\t`) character. We load this data into a list of tuples, to maintain the coupling between each document and its label.

In [5]:
def load_data(data_filename: str) -> list[tuple[str, str]]:
    """Load a two-column TSV as (language, document) tuples."""
    with open(data_filename, "r", errors="backslashreplace", encoding="utf-8") as f_in:
        observations = []
        for line in f_in:
            line = line.rstrip("\r\n")
            if not line:
                continue
            language, document = line.split("\t", 1)
            observations.append((language, document))
    return observations


### Feature Extraction

We will use ngrams as features, so we need to be able to extract them.

In [6]:
def extract_ngrams(x: str, n=3) -> "list[str]":
    """Given a string, return all character ngrams of order `n`."""
    return ["".join(s) for s in (zip(*[x[i:] for i in range(n)]))]

### Language Codes to One-Hot Vectors
And we need a function to convert a language code into a **one-hot vector** (called $\mathbf{y}$).

In [7]:
def to_onehot_vector(lang: str, langs: list[str]) -> np.ndarray:
    y = np.zeros(len(langs))
    y[langs.index(lang)] = 1
    return y

We need to be able to convert `dict`s of ngram counts to vectors of ngram counts (using a map from ngrams to dimensions of the vectors).

In [8]:
def vectorize_ngrams(counter: dict[str, int], feature_map: dict[str, int]) -> np.ndarray:
    """
    Given a dict of ngram counts and a map from features to indices, returns a vector of ngram counts.

    Parameters
    ----------
    counter : dict
        Counter/dict of ngram counts
    feature_map : dict
        Map from ngrams to indices

    Returns
    -------
    np.ndarray
        A vector of ngram counts
    """
    feature_vector = np.zeros(len(feature_map))
    for ngram, count in counter.items():
        if ngram in feature_map:
            feature_vector[feature_map[ngram]] = count
    return feature_vector

And putting together the conversion from text into ngrams, and vectorizing the ngrams:

In [9]:
def vectorize_document(document: str, feature_map: dict[str, int], ngram_length: int) -> np.ndarray:
    document_ngrams = extract_ngrams(document, ngram_length)
    vector = vectorize_ngrams(Counter(document_ngrams), feature_map)
    return vector

We need separate functions for preprocessing the training observations and the dev/test observations. The former function must return a map from language names to labels as one-hot vectors as well as a map from features (ngrams) to indices of vector dimensions.

In [10]:
def preprocess_training_observations(
    training_observations: list[tuple[str, str]], n: int = 1
) -> tuple[list[tuple[np.ndarray, np.ndarray]], dict[str, np.ndarray], dict[str, int]]:
    langs = set()
    features = set()
    obs = []

    for lang, doc in training_observations:
        langs.add(lang)
        ngrams = extract_ngrams(doc, n)
        features = features | set(ngrams)
        obs.append((lang, ngrams))
    feature_map = {feature: idx for idx, feature in enumerate(sorted(features))}
    lang_list = list(sorted(langs))
    lang_map = {lang: to_onehot_vector(lang, lang_list) for lang in lang_list}

    obs = [
        (lang_map[lang], vectorize_ngrams(Counter(ngrams), feature_map)) for (lang, ngrams) in obs
    ]

    print(f"{len(obs)} training observations.")
    return obs, lang_map, feature_map

The function for preprocessing test observations (and dev observations) takes the feature map and the language map as arguments and returns only the list of labeled observations.

In [11]:
def preprocess_test_observations(
    test_observations, feature_map: dict[str, int], lang_map: dict[str, np.ndarray], n: int = 1
) -> tuple[list[np.ndarray], list[np.ndarray]]:
    obs = []

    for i, (lang, doc) in enumerate(test_observations):
        vectorized_doc = vectorize_document(doc, feature_map, ngram_length=n)
        try:
            obs.append((lang_map[lang], vectorized_doc))
        except KeyError:
            print(f"Unkown language {lang} at index {i}. Known languages are: {lang_map.keys()}")
    print(f"{len(obs)} test observations.")
    return obs

### Classification Function

We also need to define the softmax function.

Softmax is technically

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum^K_{j=1} e^{z_j}}$$

However, if implemented naïvely, this is not numerically stable. Instead, we use:

$$\text{softmax}(z_i) = \frac{e^{z_i - \text{max}(z)}}{\sum^K_{j=1} e^{z_j - \text{max}(z)}}$$


In [12]:
def softmax(z: npt.ArrayLike) -> npt.ArrayLike:
    """Compute the softmax of a vector `z`"""
    # exp(z) can get very large. For numerical stability, we subtract a vector of very large values (np.max(z)) from z.
    return np.exp(z - np.max(z)) / np.exp(z - np.max(z)).sum()

## Training the Classifier

### Compute the gradient

For one observation, the partial derivative of the negative log-likelihood (cross-entropy) loss is

$$\frac{\partial L_{CE}}{\partial \mathbf{w}_{k,i}}
= -(\mathbf{y}_k-\hat{\mathbf{y}}_k)\mathbf{x}_i$$

where $k$ indexes the **class** (rows of $\mathbf{W}$) and $i$ indexes the **feature** (columns of $\mathbf{W}$).

Implement `grad` so that it returns the complete $K \times N$ gradient matrix. The input `x` to this function already includes the bias feature when called from your training loop.

**This is the first piece of code that you will write.**


In [13]:
def grad(W: np.ndarray, y: np.ndarray, x: np.ndarray) -> np.ndarray:
    """
    Calculate the gradient of the negative log-likelihood loss with respect to W.

    Parameters
    ----------
    W : np.ndarray
        Weight matrix of shape [K, N], where K is the number of classes.
    y : np.ndarray
        One-hot ground-truth label of shape [K].
    x : np.ndarray
        Feature vector of shape [N]. When used by the training loop, x includes
        the leading bias feature 1.

    Returns
    -------
    np.ndarray
        Gradient matrix with the same shape as W.
    """
    y_hat = softmax(W @ x)  # [K] predicted class probabilities
    # dL/dW[k, i] = -(y_k - y_hat_k) * x_i  ->  outer product gives the full [K, N] matrix
    return -np.outer(y - y_hat, x)


### Training Loop

Iterate over the training observations for `epochs` passes. At the start of each epoch, visit the observations in a new random order. Use a non-mutating shuffle:

```python
shuffled_observations = random.sample(observations, len(observations))
```

For each observation:

1. prepend a bias feature of `1` to $\mathbf{x}$;
2. compute the class probabilities $\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{W}\mathbf{x})$ as part of your gradient calculation;
3. compute the gradient;
4. update the weights using learning rate $\eta$.

At the end of every epoch, evaluate the current model on the training observations and print at least micro-precision and micro-recall. We will not grade the exact formatting of these progress messages, but they are useful for checking that training is working.

### Bias term

If the original feature vector has length $N$, initialize $\mathbf{W}$ with shape $[K, N+1]$. Column 0 contains one bias weight per class. Therefore, whenever you multiply $\mathbf{W}$ by an observation vector, prepend `1` to the vector first.

The notebook sets the random seed immediately before each graded training run. Do **not** reset the random seed inside `train`.


In [14]:
def train(observations: list[tuple[np.ndarray, np.ndarray]], eta: float, epochs: int = 1) -> np.ndarray:
    """
    Train a multinomial logistic-regression model with stochastic gradient descent.

    Parameters
    ----------
    observations : list[tuple[np.ndarray, np.ndarray]]
        Tuples of (y, x), where y is a one-hot label [K] and x is a feature
        vector [N].
    eta : float
        Learning rate.
    epochs : int
        Number of passes through the training data.

    Returns
    -------
    np.ndarray
        Trained weight matrix of shape [K, N+1], where column 0 contains biases.
    """
    K = len(observations[0][0])  # number of classes
    N = len(observations[0][1])  # number of features (without bias)
    W = np.zeros((K, N + 1))     # column 0 holds one bias weight per class

    for epoch in range(epochs):
        shuffled_observations = random.sample(observations, len(observations))
        for y, x in shuffled_observations:
            x = np.concatenate(([1.0], x))  # prepend the bias feature
            W -= eta * grad(W, y, x)

        # Evaluate the current model on the training observations
        tp, fp, fn = Counter(), Counter(), Counter()
        for y, x in observations:
            ref_lang = np.argmax(y)
            hyp_lang = np.argmax(W @ np.concatenate(([1.0], x)))
            if hyp_lang == ref_lang:
                tp[ref_lang] += 1
            else:
                fp[hyp_lang] += 1
                fn[ref_lang] += 1
        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"micro-P {micro_precision(tp, fp):.3f}, "
            f"micro-R {micro_recall(tp, fn):.3f}, "
            f"macro-F1 {macro_f1(tp, fp, fn):.3f}"
        )

    return W


## Classification

The classification function is very simple.

In [15]:
def classify(W: np.ndarray, x: np.ndarray) -> np.intp:
    """
    Return the index of the most probable language.

    Parameters
    ----------
    W : np.ndarray
        Weight matrix with one row per class and a bias in column 0.
    x : np.ndarray
        Real-valued feature vector without the bias feature.

    Returns
    -------
    np.intp
        Index of the predicted class.
    """
    x = np.concatenate(([1.0], x))  # prepend the bias feature, as in training
    # softmax is monotonic, so the class with the highest score is also the most probable
    return np.argmax(W @ x)


## Evaluate the Classifier

Then, a function to train and evaluate the classifier.

In [16]:
def evaluate(train_set, test_set, eta, epochs=3):
    """Trains and evaluates a model."""
    print("Training model")
    W = train(train_set, eta, epochs=epochs)
    print("\nCLASSIFY")
    tp, fp, fn = Counter(), Counter(), Counter()
    for ref_lang_vec, x in test_set:
        ref_lang = np.argmax(ref_lang_vec)

        hyp_lang = classify(W, x)
        if hyp_lang == ref_lang:
            tp[ref_lang] += 1
        else:
            fp[hyp_lang] += 1
            fn[ref_lang] += 1
    # Print metrics

    test_macro_f1 = macro_f1(tp, fp, fn)
    test_micro_f1 = micro_f1(tp, fp, fn)
    print(f"macro-averaged F1:\t\t{test_macro_f1:.3f}")
    print(f"micro-averaged F1:\t\t{test_micro_f1:.3f}")
    return test_macro_f1, test_micro_f1

### Putting it all together

Load the data:

In [17]:
train_observations = load_data("train.tsv")
test_observations = load_data("test.tsv")

### Inspecting the data

Before training, inspect the dataset and verify your assumptions.

For both the training and test sets, output a dictionary mapping each language label to its number of observations. Sort by **descending count**, breaking ties **alphabetically by label**.

Example format:

```python
{"hausa": 4000, "indonesian": 3500, ...}
```

Think about:
- How many training and test instances are there?
- Are the same labels present in both splits?
- Is the class distribution balanced?


In [18]:
# this cell's output will be used for test 1.1
# your code here - train set
def label_counts(observations: list[tuple[str, str]]) -> dict[str, int]:
    """Count observations per language, sorted by descending count, then alphabetically."""
    counts = Counter(lang for lang, _ in observations)
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))


print(label_counts(train_observations))

{'swahili': 419, 'tagalog': 384, 'manobo': 111, 'nahuatl': 98, 'hausa': 94, 'indonesian': 94}


In [19]:
# this cell's output will be used for test 1.2
# your code here - test set
print(label_counts(test_observations))

{'swahili': 140, 'tagalog': 128, 'manobo': 37, 'hausa': 32, 'nahuatl': 32, 'indonesian': 31}


### Set hyperparameters and parameters

Before training the model, we have to set the learning rate $\eta$ and the order of the ngrams used in feature extraction.

In [20]:
eta = 0.0005  # Do not change this.
epochs = 4    # Do not change this.
ASSIGNMENT_SEED = 2028  # Do not change this.


In [21]:
order_of_ngrams = 3 # Change this.

### Running our training and evaluation loops

Now train and evaluate your classifier using each character $n$-gram order in

$$n \in \{1,2,3,4\}.$$

Record the macro-F1 and micro-F1 for each value. Select the model with the **highest macro-F1**; use micro-F1 only as a tie-breaker. Before submission, set `order_of_ngrams` to your selected value and rerun the cells below.


In [22]:
train_set, lang_map, feature_map = preprocess_training_observations(
    train_observations, n=order_of_ngrams
)
test_set = preprocess_test_observations(
    test_observations, feature_map, lang_map, n=order_of_ngrams
)

random.seed(ASSIGNMENT_SEED)
test_macro_f1, test_micro_f1 = evaluate(train_set, test_set, eta, epochs=epochs)


1200 training observations.
400 test observations.
Training model
Epoch 1/4: micro-P 0.955, micro-R 0.955, macro-F1 0.932
Epoch 2/4: micro-P 0.981, micro-R 0.981, macro-F1 0.976
Epoch 3/4: micro-P 0.991, micro-R 0.991, macro-F1 0.990
Epoch 4/4: micro-P 0.993, micro-R 0.993, macro-F1 0.993

CLASSIFY
macro-averaged F1:		0.986
micro-averaged F1:		0.988


Print the tuple `(macro_f1, micro_f1)` for your **selected** $n$-gram order. Then explain:

1. Which value of $n$ did you select?
2. What did you observe across $n=1,2,3,4$?
3. Why might very small or larger values of $n$ perform differently on this dataset?

**Your response:** I decided to select n = 3, or trigrams, due to the macro-F1 = 0.986 and micro-F1 = 0.988 being jointly the highest numbers in testing. When I was running each of n element of {1,2,3,4}, I noticed that the testing accuracy measures created a large jump from the unigram to bigrams. The trigram was then slightly the best before a decline by the fourth n-gram. I also put this through the test cell output finding a higher joint macro and micro F1-score for the trigram. Now, as to why this might be occurring, the n-grams are used as classification features. Each document is represented as a bag of the n-gram counts. The multinomial logistic regression we are working with in this notebook would learn one weight per (language, n-gram) pair. Now, digging deeper into the data and this concept discussed in class, you can see when n=4, there are a lot more features. The training budget for that instance then grows and the model right now isn't well fit for that increase. The learning rate consideration at trigrams seems to be the best fit. At n=4, there could be too much sparsity in the data that leaves a lot of the data filled with 0's that means it is hard in to update the weights using stochastic gradient descent. Also, as the n increases from 1 to 2 and to 3, we can see that the features begin to capture what is necessary about these language types such as the phonotactics and start to determine the general word-boundary and affix patterns since spaces would also be apart of these n-grams. 

In [23]:
# the output of this cell will be used for test 2.1
print((test_macro_f1, test_micro_f1))

(0.9863670553352395, 0.9875)


## Inspecting Classification Results

Aggregate metrics tell us how well a classifier performs overall, but they do not explain *which* examples it gets wrong or why.

For the remainder of the assignment, use a newly trained **trigram** classifier trained for exactly **one epoch**. This fixed, intentionally under-trained model produces enough errors to support a useful error analysis while still performing reasonably well.


In [24]:
# Re-train a fixed trigram classifier for inspection. Do not change these values.
INSPECTION_NGRAMS = 3
INSPECTION_EPOCHS = 1

train_set, lang_map, feature_map = preprocess_training_observations(
    train_observations, n=INSPECTION_NGRAMS
)
test_set = preprocess_test_observations(
    test_observations, feature_map, lang_map, n=INSPECTION_NGRAMS
)

random.seed(ASSIGNMENT_SEED)
W_inspect = train(train_set, eta, epochs=INSPECTION_EPOCHS)

tp, fp, fn = Counter(), Counter(), Counter()
for ref_lang_vec, x in test_set:
    ref_lang = np.argmax(ref_lang_vec)
    hyp_lang = classify(W_inspect, x)
    if hyp_lang == ref_lang:
        tp[ref_lang] += 1
    else:
        fp[hyp_lang] += 1
        fn[ref_lang] += 1

print(f"macro-averaged F1:\t\t{macro_f1(tp, fp, fn):.3f}")
print(f"micro-averaged F1:\t\t{micro_f1(tp, fp, fn):.3f}")


1200 training observations.
400 test observations.
Epoch 1/1: micro-P 0.955, micro-R 0.955, macro-F1 0.932
macro-averaged F1:		0.920
micro-averaged F1:		0.945


### Writing a prediction function

Write a function that takes a list of observations, and produces a list of either class indices, or class names based on a parameter. This will allow you both to look at individual results from evaluating on an existing set of data (like the test set), but also for you to evaluate your classifier on new data (i.e. any string). This will involve vectorizing the list of documents, and classifying those vectors. Once that's complete, construct an inverse language mapping, from predicted indices to the language they represent.

In [25]:
def predict(
    documents: list[str],
    W: np.ndarray,
    feature_map: dict[str, int],
    lang_map: dict[str, np.ndarray],
    ngram_length: int,
    return_class_names=False,
):
    # your code here
    predictions = [
        classify(W, vectorize_document(document, feature_map, ngram_length))
        for document in documents
    ]
    if return_class_names:
        # Inverse language map: class index -> language name
        inverted_lang_map = {np.argmax(onehot): lang for lang, onehot in lang_map.items()}
        return [inverted_lang_map[idx] for idx in predictions]
    return predictions

Now, for each instance in the test set, print a tuple of the actual label, then the predicted label.

In [26]:
# the output of this cell will be used for test 3.1
test_labels, test_documents = zip(*test_observations)
test_predictions = predict(
    test_documents,
    W_inspect,
    feature_map,
    lang_map,
    ngram_length=INSPECTION_NGRAMS,
    return_class_names=True,
)

print(list(zip(test_labels, test_predictions)))

[('swahili', 'swahili'), ('tagalog', 'tagalog'), ('tagalog', 'tagalog'), ('nahuatl', 'nahuatl'), ('swahili', 'swahili'), ('tagalog', 'tagalog'), ('tagalog', 'tagalog'), ('tagalog', 'tagalog'), ('tagalog', 'tagalog'), ('swahili', 'swahili'), ('swahili', 'swahili'), ('swahili', 'swahili'), ('tagalog', 'tagalog'), ('hausa', 'swahili'), ('swahili', 'swahili'), ('swahili', 'swahili'), ('swahili', 'swahili'), ('hausa', 'hausa'), ('indonesian', 'indonesian'), ('swahili', 'swahili'), ('tagalog', 'tagalog'), ('nahuatl', 'nahuatl'), ('swahili', 'swahili'), ('indonesian', 'indonesian'), ('tagalog', 'tagalog'), ('tagalog', 'tagalog'), ('manobo', 'manobo'), ('indonesian', 'indonesian'), ('swahili', 'swahili'), ('swahili', 'swahili'), ('swahili', 'swahili'), ('swahili', 'swahili'), ('tagalog', 'tagalog'), ('manobo', 'tagalog'), ('tagalog', 'tagalog'), ('swahili', 'swahili'), ('tagalog', 'tagalog'), ('nahuatl', 'nahuatl'), ('tagalog', 'tagalog'), ('swahili', 'swahili'), ('tagalog', 'tagalog'), ('taga

### Plotting a Confusion Matrix

A *confusion matrix* is a $K \times K$ matrix in which cell $(i,j)$ counts the number of examples whose true class is $i$ and whose predicted class is $j$. Diagonal entries are correct classifications; off-diagonal entries are errors.

![A confusion matrix for a binary classification problem](confusion_matrix.png)

Write a function that returns a NumPy confusion matrix for the supplied labels and predictions. Use the class index encoded by `lang_map` for both rows and columns. Because `preprocess_training_observations` sorts language names before building the map, this also gives a deterministic class order.


In [27]:
def get_confusion_matrix(
    labels: list[str], predictions: list[str], lang_map: dict[str, np.ndarray]
) -> np.ndarray:
    # your code here
    K = len(lang_map)
    confusion_matrix = np.zeros((K, K), dtype=int)
    for label, prediction in zip(labels, predictions):
        i = np.argmax(lang_map[label])       # row: true class
        j = np.argmax(lang_map[prediction])  # column: predicted class
        confusion_matrix[i, j] += 1
    return confusion_matrix

Now, print your confusion matrix.

In [28]:
# the output of this cell will be used for test 3.2
get_confusion_matrix(test_labels, test_predictions, lang_map)

array([[ 17,   0,   0,   0,  15,   0],
       [  0,  30,   0,   0,   0,   1],
       [  0,   0,  35,   0,   0,   2],
       [  0,   0,   0,  29,   3,   0],
       [  0,   0,   0,   0, 140,   0],
       [  0,   0,   0,   0,   1, 127]])

From your confusion matrix, how many **Manobo** examples are misclassified as **Tagalog**? Print each of those misclassified documents on a separate line.


In [29]:
# the output of this cell will be used for test 3.3
# your code here
# Count comes from the confusion matrix (row = true manobo, column = predicted tagalog).
confusion_matrix = get_confusion_matrix(test_labels, test_predictions, lang_map)
manobo_idx = np.argmax(lang_map["manobo"])
tagalog_idx = np.argmax(lang_map["tagalog"])
n_manobo_as_tagalog = confusion_matrix[manobo_idx, tagalog_idx]  # -> 2

# Print only the misclassified documents, one per line.
for label, prediction, document in zip(test_labels, test_predictions, test_documents):
    if label == "manobo" and prediction == "tagalog":
        print(document)


no anak ni Miliya, no anak ni Maynan, no anak ni Matata, no anak ni Natan, no anak ni David,
no anak ni Sirug, no anak ni Ragaw, no anak ni Pilig, no anak ni Ibir, no anak ni Sala,


The fixed inspection model should reveal a small set of Manobo examples that are predicted as Tagalog.

Formulate a hypothesis for why these examples are confused. Inspect the misclassified documents, the training data, and reliable resources about Manobo and Tagalog. Consider both:

- linguistic/data reasons (for example, related languages, shared strings, genre, names, or repeated constructions), and
- machine-learning reasons (feature overlap, class imbalance, sparse features, limited training, etc.).

Propose **2-3 concrete experiments** that could test your hypothesis.

**Your response:** The fixed trigram inspection model misclassifies 2 of the Manobo test documents as Tagalog. Both appear to be of similar construction relatively. My hypothesis is that these documents have character trigrams that relate to the grammatical morphemes that are actually considerably similar between Manobo and Tagalog, since they share relation to the Austronesian languages. The classifier has incorrectly learned to treat this shared material as Tagalog since it is a larger class of the data. Overall, there are many areas where they share similar vocabulary, case-marking, etc. For a proposed experiment, we can consider morpheme ablation. We could take the two misclassified documents and remove those shared morphemes "anak", "ni", etc. Leaving the rest of that text unchanged, we can then re-vectorize and re-classify each version with the same inspection model. Assuming the hypothesis is right, this will likely flip this classification from Tagalog to Manobo, otherwise, there is no cause between this instrumental removal. For another experiment, we could do class re-balancing by fixing the trigram model with a training set that is downsampled in the Tagalog amounts to have it matching Manobo. This could generally increase the ability to differentiate equally between the two better than before keeping all else equal in the model. If the errors persist, the class imbalance can't be considered the full correlatory measure that is causing these errors. Finally, for the last experiment, I would propose doing a phonotactics only control. Here, we would map every character that is in the corpus to C or V (consonant, vowel, with spaces preserved), which then removes the lexicon but preserves that syllable structure. From here, we could compare the character n-gram profiles between all the language pairs, and if this is informative, train the classifier on this transformed text. This considers then we can see that shared closeness in measurement between Tagalog and Manobo. If all six languages seem to look the same in C/V terms, then the syllable structure cannot explain a confusion specific to this pair, and then the shared-morphemes from experiment one would be more likely favored as our reasoning. 


### Examining Feature Weights

The $K \times (N+1)$ weight matrix `W_inspect` can be interpreted one row at a time: each row contains the weights for one language class. Column 0 is the bias, while columns 1 through $N$ correspond to the $N$ entries in `feature_map`.

Construct an inverted feature map from vector index to trigram. Then extract the **Manobo feature-weight vector**, excluding the bias column, and print its shape.

**Important:** Use `W_inspect` from the fixed trigram inspection model above. If feature index `i` refers to an entry in `feature_map`, its weight is in column `i + 1` of `W_inspect`.


In [30]:
# the output of this cell will be used for test 4.1
# Invert the feature map
inverted_feature_map = {v: k for k, v in feature_map.items()}

# your code here
manobo_idx = np.argmax(lang_map["manobo"])
manobo_weights = W_inspect[manobo_idx, 1:]  # drop column 0 (bias) so index i matches feature_map
print(manobo_weights.shape)

(7710,)


Find the 10 features with the largest positive weights for **Manobo**. Print one tuple `(feature_name, feature_weight)` per line, from largest to smallest weight.

Example:

```text
("aaa", 0.04597442104739769)
("bbb", 0.03454984682736487)
```


In [31]:
# the output of this cell will be used for test 4.2
# your code here
def top_features(weights: np.ndarray, k: int = 10) -> list[tuple[str, float]]:
    """Return the k (feature_name, weight) pairs with the largest weights, in descending order."""
    top_indices = np.argsort(-weights, kind="stable")[:k]
    return [(inverted_feature_map[i], float(weights[i])) for i in top_indices]


for feature in top_features(manobo_weights):
    print(feature)

(' to', 0.1137209396518992)
('to ', 0.11138494147168929)
(' no', 0.06335478966724993)
('no ', 0.05114197961516186)
('n t', 0.0414804375531509)
('an ', 0.03613409783348637)
(' og', 0.03533720734699043)
('ow ', 0.02932469245152898)
('din', 0.02859629966971515)
('on ', 0.028192673333510394)


Now repeat the analysis for **every language**. For each language, print the language name and then a list of its 10 highest-weight trigrams, in descending weight order.

Example:

```text
hausa
[("aaa", 0.0459), ...]
indonesian
[("bbb", 0.0412), ...]
```

Remember to exclude the bias column from the feature ranking.


In [32]:
# the output of this cell will be used for test 4.3
# your code here
for lang, onehot in lang_map.items():
    lang_weights = W_inspect[np.argmax(onehot), 1:]  # exclude the bias column
    print(lang)
    print(top_features(lang_weights))

hausa
[('da ', 0.060858422298388344), (' da', 0.05726460612924964), ('in ', 0.034395494830984356), (' sh', 0.02579738483838394), (' su', 0.02255704502236063), ('ar ', 0.021417086815594507), ('ai ', 0.019894599964089803), (' ba', 0.019629062716936804), ('ka ', 0.019619746861785824), (' ga', 0.019576247627961712)]
indonesian
[('an ', 0.07320286895955072), (' me', 0.06395261460918851), ('ang', 0.04464637437739603), (' se', 0.03926429615434204), ('ber', 0.03776141071847242), (' be', 0.037613006625085446), ('men', 0.03315871130927872), (' ke', 0.03259631210312764), ('eng', 0.03203539955087078), ('ran', 0.03129907868369211)]
manobo
[(' to', 0.1137209396518992), ('to ', 0.11138494147168929), (' no', 0.06335478966724993), ('no ', 0.05114197961516186), ('n t', 0.0414804375531509), ('an ', 0.03613409783348637), (' og', 0.03533720734699043), ('ow ', 0.02932469245152898), ('din', 0.02859629966971515), ('on ', 0.028192673333510394)]
nahuatl
[('tla', 0.07357283450626226), (' tl', 0.06179772484467821

Choose one language and consult its Wikipedia page or another reliable source. Do its top-10 trigrams make sense in light of what you learned about the language? Explain why or why not, considering both linguistic properties and how multinomial logistic regression learns weights from this particular dataset.

**Your response:** I decided to choose the Tagalog grammar page on Wikipedia. Its ten highest-weighted trigrams in W_inspect are 'ng ', 'ang', 'sa ', ' sa', ' ng', ' an', 'ay ', ' na', 'at ', and 'g m'. These make sense linguistically, and they also reflect how multinomial logistic regression assigns weights on this particular dataset. According to the Wikipedia article on Tagalog grammmar, Tagalog marks syntactic role of noun phrases with a small se tof case markers that precede the phrase, such as "ang". Here, it is a direct case marker that marks the noun which has a special relation to the verb in its clause usage. Furthermore, "ng" is an indirect case marker which marks the location, beneficiary, instrument, and any other oblique argument. This source also described "ay" as an inversion marker that is used when a constituent is fronted, "na/-ng" as linkers that connect a modifier to the word it modifies, and "mga" as the plural marker for common nouns. What this indicates to what we saw earlier, is that those mentioned above are typically separate markers that occur before nearly every noun phrase and generate character trigrams with leading or trailing spaces. This appears to be exactly what the model had learned in our notebook run. For instance "ang" and "ng", are ' an', 'ng ', and ' ng'. This can be determined with the other examples relatively as well. The distribution of these increased features is what drives the stochastic gradient descent update of the weights in the model. These trigrams that are frequently appearing in the Tagalog documents and are seemingly rare in other places can allow for this better distinction of classes. Overall, none of these top features are a content based word, which fits this grammatical markers interpretation that they occur far more often than the respective nouns and verbs of Tagalog. 


References: "Tagalog grammar," Wikipedia, https://en.wikipedia.org/wiki/Tagalog_grammar (accessed September 22, 2026).


### Extra Credit

Implement at least one experiment that tests your hypothesis about the Manobo-to-Tagalog errors. To receive credit, include:

- your hypothesis,
- the experiment and why it is an appropriate test,
- the result, and
- a short interpretation of what the result supports or fails to support.

Credit is based on the quality of the hypothesis, experimental design, evidence, and explanation rather than on whether the hypothesis turns out to be correct.


**My Answer**:
I would like to experiment on the shared morphemes that was previously discussed as an experiment in the earlier questions to showcase this error in classification mechanism. 

My hypothesis is that the Manobo and Tagalog errors are most influenced by shared Austronesian grammatical morphemes. This effect would appear in those documents that are dominated by these morphemes. 

I will experiment on the difference that can be seen by downsampling Tagalog to match a similar training size input to reduce increased preference, and also perform morpheme ablation to delete "anak" and "ni" that were increasingly prevalent in earlier analysis and assess the re-classification output correctness. 

In [33]:
# Implement extra credit here.
MANOBO = np.argmax(lang_map["manobo"])
TAGALOG = np.argmax(lang_map["tagalog"])
LANGS = sorted(lang_map)


def tokenize(document: str) -> list[str]:
    """Split into lowercase alphabetic words, discarding punctuation."""
    return "".join(c if c.isalpha() else " " for c in document.lower()).split()

def doc_rate(word: str, observations: list[tuple[str, str]]) -> dict[str, float]:
    """Percent of each language's documents that contain `word` as a whole word."""
    hits, totals = Counter(), Counter()
    for lang, doc in observations:
        totals[lang] += 1
        if word in tokenize(doc):
            hits[lang] += 1
    return {lang: 100 * hits[lang] / totals[lang] for lang in totals}

print("A. Shared Austronesian vocabulary: % of TRAINING documents containing each word")
print(f"{'word':<6}" + "".join(f"{lang:>12}" for lang in LANGS))
for word in ["anak", "ni", "si"]:
    rate = doc_rate(word, train_observations)
    print(f"{word:<6}" + "".join(f"{rate[lang]:11.0f}%" for lang in LANGS))

errors = [
    doc for label, pred, doc in zip(test_labels, test_predictions, test_documents)
    if label == "manobo" and pred == "tagalog"
]
print(f"\nB. Manobo test documents predicted as Tagalog: {len(errors)}")

def margin(document: str, W: np.ndarray, features: dict[str, int], lang_idx: dict[str, int]) -> float:
    """score(tagalog) - score(manobo); positive means the document is called Tagalog."""
    x = np.concatenate(([1.0], vectorize_document(document, features, INSPECTION_NGRAMS)))
    scores = W @ x
    return scores[lang_idx["tagalog"]] - scores[lang_idx["manobo"]]

ORIGINAL_IDX = {lang: np.argmax(onehot) for lang, onehot in lang_map.items()}

# Which trigrams produce the margin in the original documents?
for i, doc in enumerate(errors, 1):
    x = vectorize_document(doc, feature_map, INSPECTION_NGRAMS)
    contribution = (W_inspect[TAGALOG, 1:] - W_inspect[MANOBO, 1:]) * x
    ranked = np.argsort(-contribution)
    print(f"  doc {i}: margin {margin(doc, W_inspect, feature_map, ORIGINAL_IDX):+.3f}")
    print(f"    most pro-tagalog: {[(inverted_feature_map[j], round(float(contribution[j]), 3)) for j in ranked[:4]]}")
    print(f"    most pro-manobo : {[(inverted_feature_map[j], round(float(contribution[j]), 3)) for j in ranked[-3:]]}")

def drop_words(document: str, targets: set[str]) -> str:
    """Delete whole-word occurrences of `targets`, keeping every other token as-is."""
    kept = [tok for tok in document.split() if "".join(c for c in tok if c.isalpha()).lower() not in targets]
    return " ".join(kept)

def drop_names(document: str) -> str:
    """Delete capitalized tokens (the proper names in these genealogies)."""
    kept = [tok for tok in document.split() if not "".join(c for c in tok if c.isalpha())[:1].isupper()]
    return " ".join(kept)

CONDITIONS = {
    "original                      ": lambda d: d,
    "delete 'anak' (shared)        ": lambda d: drop_words(d, {"anak"}),
    "delete 'ni'   (shared)        ": lambda d: drop_words(d, {"ni"}),
    "delete both   (shared)        ": lambda d: drop_words(d, {"anak", "ni"}),
    "delete 'no'   (Manobo marker) ": lambda d: drop_words(d, {"no"}),
    "delete names  (control)       ": drop_names,
}

print("\nC. Morpheme ablation under the ORIGINAL (imbalanced) model")
print("   margin = score(tagalog) - score(manobo); negative = classified Manobo (correct)")
print(f"   {'condition':<31}" + "".join(f"{'doc ' + str(i):>17}" for i in (1, 2)))
for name, transform in CONDITIONS.items():
    row = []
    for doc in errors:
        m = margin(transform(doc), W_inspect, feature_map, ORIGINAL_IDX)
        row.append(f"{m:+.3f} ({'TAGALOG' if m > 0 else 'manobo'})")
    print(f"   {name}" + "".join(f"{c:>17}" for c in row))

def downsample(observations: list[tuple[str, str]], lang: str, cap: int, rng) -> list[tuple[str, str]]:
    """Keep only `cap` randomly chosen documents of `lang`; leave other languages intact."""
    indices = [i for i, (l, _) in enumerate(observations) if l == lang]
    keep = set(rng.sample(indices, cap))
    return [obs for i, obs in enumerate(observations) if obs[0] != lang or i in keep]

MANOBO_TRAIN_SIZE = sum(1 for lang, _ in train_observations if lang == "manobo")
REPEATS = 5
balanced_margins = {name: [] for name in CONDITIONS}
balanced_errors = []

print(f"\nD. Retrain with Tagalog downsampled to {MANOBO_TRAIN_SIZE} docs, {REPEATS} random subsamples")
for repeat in range(REPEATS):
    subsample = downsample(train_observations, "tagalog", MANOBO_TRAIN_SIZE, random.Random(1000 + repeat))
    bal_train, bal_lang_map, bal_features = preprocess_training_observations(
        subsample, n=INSPECTION_NGRAMS
    )
    random.seed(ASSIGNMENT_SEED)  # same seed and budget as the graded inspection run
    W_balanced = train(bal_train, eta, epochs=INSPECTION_EPOCHS)

    bal_idx = {lang: np.argmax(onehot) for lang, onehot in bal_lang_map.items()}
    predictions = predict(
        test_documents, W_balanced, bal_features, bal_lang_map,
        ngram_length=INSPECTION_NGRAMS, return_class_names=True,
    )
    n_errors = sum(
        1 for label, pred in zip(test_labels, predictions) if label == "manobo" and pred == "tagalog"
    )
    balanced_errors.append(n_errors)
    for name, transform in CONDITIONS.items():
        balanced_margins[name].append(
            np.mean([margin(transform(doc), W_balanced, bal_features, bal_idx) for doc in errors])
        )
    print(f"   subsample {repeat + 1}: manobo->tagalog test errors = {n_errors}")

print("\nE. Mean margin over the two documents (negative = classified Manobo)")
print(f"   {'condition':<31}{'imbalanced':>13}{'balanced':>11}{'effect':>10}")
for name, transform in CONDITIONS.items():
    imbalanced = np.mean([margin(transform(doc), W_inspect, feature_map, ORIGINAL_IDX) for doc in errors])
    balanced = np.mean(balanced_margins[name])
    print(f"   {name}{imbalanced:+13.3f}{balanced:+11.3f}{balanced - imbalanced:+10.3f}")

print(f"\n   manobo->tagalog errors: original model = {len(errors)}, "
      f"balanced models = {balanced_errors} (mean {np.mean(balanced_errors):.1f})")

A. Shared Austronesian vocabulary: % of TRAINING documents containing each word
word         hausa  indonesian      manobo     nahuatl     swahili     tagalog
anak            0%         11%          5%          0%          0%          9%
ni              7%          0%         24%          0%         14%         28%
si              0%          1%         28%          0%          1%         18%

B. Manobo test documents predicted as Tagalog: 2
  doc 1: margin +0.184
    most pro-tagalog: [(' an', 0.386), (' ni', 0.138), ('o a', 0.094), ('ata', 0.047)]
    most pro-manobo : [('ana', -0.031), ('no ', -0.316), (' no', -0.335)]
  doc 2: margin +0.123
    most pro-tagalog: [(' an', 0.386), (' ni', 0.138), ('o a', 0.094), ('k n', 0.044)]
    most pro-manobo : [('ana', -0.031), ('no ', -0.316), (' no', -0.335)]

C. Morpheme ablation under the ORIGINAL (imbalanced) model
   margin = score(tagalog) - score(manobo); negative = classified Manobo (correct)
   condition                               

Overall, I would need a much more rigorous design of the difference to determine causality (especially working with only the two given documents). So, I preferred a more simpler margin based correlative approach that assesses the difference in classification before and after the changes of removing these words and also downsampling. Some other limitations include that downsampling may not be the most effective equal measure for this data and wasn't fully verified. Furthermore, deleting a given word also deletes the related boundary trigrams that would be around it. 

With that in mind, it does seem to suggest from the results that removing these morphemes supports classification of Manobo over Tagolog for this given instance then as we see those associated negative values in the margins of those tests. Furthermore, it did seem that downsampling helped generally remove the errors as well in the samples. The procedure seems to generally respond based on what was seen in the control tests as well to have them move in the correct fashion positive or negative based on what was removed. Furthermore, removing the names barely affects the prediction. Overall, it seems that both interventions have a joint increased effect in finding a better general true classification output by accounting for these document morpheme similarities and data input size differences. 

